## model 

In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from typing import Annotated
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict
from langgraph.graph import MessagesState
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()

#llm
##################

llm = init_chat_model("ollama:nemotron-3-super:cloud",base_url="https://ollama.com")
llmg = init_chat_model("google_genai:gemini-2.5-flash")

response = llmg.invoke("What is the color of the sky answer in one word?")
print(response.content)

Blue


## embedding model

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_chroma import Chroma

In [3]:
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

# Initialize the model (downloads automatically on first run)
model_name = "BAAI/bge-m3"
model_kwargs = {"device": "cpu"}  # Change to "cuda" if you have a Nvidia GPU
encode_kwargs = {"normalize_embeddings": True}

embeddings = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1475.26it/s]


## croma db instance

In [5]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="C:\E_DRIVE\Langraph-Refresher\database\example\chroma_db",
)

## CURD operation to DB

### add Data

In [7]:
from uuid import uuid4

from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
    id=1,
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
    id=2,
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
    id=3,
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
    id=4,
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
    id=5,
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
    id=6,
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
    id=7,
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
    id=8,
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
    id=9,
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
    id=10,
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]
uuids = [str(uuid4()) for _ in range(len(documents))]

vector_store.add_documents(documents=documents, ids=uuids)

['794dcc54-b3f3-4563-9ee8-4d1495482de7',
 'dc508371-b19b-40c6-93cf-3ca54406cd39',
 '1a742e1e-16a5-4f4f-9ccd-c92231e8eb94',
 'c2c42589-486e-4d49-a8d1-fe56f073a7a0',
 '109ecca0-5701-40fb-9cf1-c8daa8003b0a',
 'e8d5d79f-1a78-405e-858e-93a8af5022ca',
 'a90d80f3-043f-4b91-9873-0f86007c9eaa',
 '8d12512b-4970-4b9b-965d-60b8d4800a7b',
 '569db147-fda7-4c91-b051-6ae4a4378d2a',
 '84d9fd10-c910-48df-8877-f49166dffe7d']

### update data

In [ ]:
updated_document_1 = Document(
    page_content="I had chocolate chip pancakes and fried eggs for breakfast this morning.",
    metadata={"source": "tweet"},
    id=1,
)

updated_document_2 = Document(
    page_content="The weather forecast for tomorrow is sunny and warm, with a high of 82 degrees.",
    metadata={"source": "news"},
    id=2,
)

vector_store.update_document(document_id=uuids[0], document=updated_document_1)
# You can also update multiple documents at once
vector_store.update_documents(
    ids=uuids[:2], documents=[updated_document_1, updated_document_2]
)

### retrive data

In [9]:
results = vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=2,
    filter={"source": "tweet"},
)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

* Building an exciting new project with LangChain - come check it out! [{'source': 'tweet'}]
* LangGraph is the best framework for building stateful, agentic applications! [{'source': 'tweet'}]


In [10]:
results = vector_store.similarity_search_by_vector(
    embedding=embeddings.embed_query("I love green eggs and ham!"), k=1
)
for doc in results:
    print(f"* {doc.page_content} [{doc.metadata}]")

* I had chocolate chip pancakes and scrambled eggs for breakfast this morning. [{'source': 'tweet'}]


In [11]:
# Returns a list of (Document, float) tuples
results_with_scores = vector_store.similarity_search_with_score(
   "LangChain provides abstractions to make working with LLMs easy",
    k=2,
    filter={"source": "tweet"}
)

for doc, score in results_with_scores:
    # Note: Chroma uses L2/Euclidean distance by default.
    # A LOWER score means the text is MORE similar to your query.
    print(f"Distance Score: {score}") 
    print(f"Content: {doc.page_content}\n")

Distance Score: 0.7377016544342041
Content: Building an exciting new project with LangChain - come check it out!

Distance Score: 0.9178253412246704
Content: LangGraph is the best framework for building stateful, agentic applications!



In [12]:
retriever = vector_store.as_retriever(
    search_type="mmr", search_kwargs={"k": 1, "fetch_k": 5}
)
retriever.invoke("Stealing from the bank is a crime", filter={"source": "news"})

[Document(id='c2c42589-486e-4d49-a8d1-fe56f073a7a0', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]

### retrive ids of insterted chunks

In [16]:
# Fetch all items (or a limited number using the limit parameter)
db_data = vector_store.get(limit=10)

# Extract the IDs list
all_ids = db_data["ids"]
print("First 5 IDs in the DB:", all_ids[:5])

First 5 IDs in the DB: ['794dcc54-b3f3-4563-9ee8-4d1495482de7', 'dc508371-b19b-40c6-93cf-3ca54406cd39', '1a742e1e-16a5-4f4f-9ccd-c92231e8eb94', 'c2c42589-486e-4d49-a8d1-fe56f073a7a0', '109ecca0-5701-40fb-9cf1-c8daa8003b0a']


In [20]:
# Pass a list containing the ID(s) you want to fetch
target_ids = ['794dcc54-b3f3-4563-9ee8-4d1495482de7', 'dc508371-b19b-40c6-93cf-3ca54406cd39']

documents = vector_store.get_by_ids(all_ids)

for doc in documents:
    print(doc)

page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.' metadata={'source': 'tweet'}
page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.' metadata={'source': 'news'}
page_content='Building an exciting new project with LangChain - come check it out!' metadata={'source': 'tweet'}
page_content='Robbers broke into the city bank and stole $1 million in cash.' metadata={'source': 'news'}
page_content='Wow! That was an amazing movie. I can't wait to see it again.' metadata={'source': 'tweet'}
page_content='Is the new iPhone worth the price? Read this review to find out.' metadata={'source': 'website'}
page_content='The top 10 soccer players in the world right now.' metadata={'source': 'website'}
page_content='LangGraph is the best framework for building stateful, agentic applications!' metadata={'source': 'tweet'}
page_content='The stock market is down 500 points today due to fears of a recession.' metadata={'s

### delete data

In [ ]:
# Delete the health insurance chunk ('doc_chunk_1')
vector_store.delete(ids=["doc_chunk_1"])
print("\n--- DELETE: Removed 'doc_chunk_1' from DB. ---")

## DB from the pdf documnennt

### load PDF doc

In [22]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("resources/acmecorp-employee-handbook.pdf")
data = loader.load()
print(data)

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-11-20T23:23:16+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-11-20T23:23:16+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'resources/acmecorp-employee-handbook.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Employee Handbook\nNon-Disclosure Agreement (NDA) Policy\nEmployees must protect confidential information belonging to the company, its clients, and partners.\nThis includes, but is not limited to, product roadmaps, customer data, internal communications,\nproprietary algorithms, financial information, and unreleased features. Confidential information may not\nbe shared with unauthorized individuals inside or outside the organization. These obligations continue\nafter employment ends.\nWorkplace Conduct Policy\nEmployees must maintain a respectful, professional environment

In [ ]:
data[0].metadata

{'producer': 'ReportLab PDF Library - www.reportlab.com',
 'creator': '(unspecified)',
 'creationdate': '2025-11-20T23:23:16+00:00',
 'author': '(anonymous)',
 'keywords': '',
 'moddate': '2025-11-20T23:23:16+00:00',
 'subject': '(unspecified)',
 'title': '(anonymous)',
 'trapped': '/False',
 'source': 'resources/acmecorp-employee-handbook.pdf',
 'total_pages': 1,
 'page': 0,
 'page_label': '1'}

In [31]:
print(data[0].page_content)

Employee Handbook
Non-Disclosure Agreement (NDA) Policy
Employees must protect confidential information belonging to the company, its clients, and partners.
This includes, but is not limited to, product roadmaps, customer data, internal communications,
proprietary algorithms, financial information, and unreleased features. Confidential information may not
be shared with unauthorized individuals inside or outside the organization. These obligations continue
after employment ends.
Workplace Conduct Policy
Employees must maintain a respectful, professional environment free from harassment, discrimination,
and intimidation. All employees are expected to follow organizational values, collaborate effectively,
and communicate constructively. Disruptive behavior, verbal abuse, or misuse of company systems is
prohibited. Violations may result in disciplinary action.
Paid Time Off (PTO) Policy
Full■time employees accrue PTO according to the following schedule:  0–1 years of service: 10 days
per

### chunking of the doc

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)

all_splits = text_splitter.split_documents(data)

print(len(all_splits))

3


In [33]:
all_splits

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-11-20T23:23:16+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-11-20T23:23:16+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'resources/acmecorp-employee-handbook.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'start_index': 0}, page_content='Employee Handbook\nNon-Disclosure Agreement (NDA) Policy\nEmployees must protect confidential information belonging to the company, its clients, and partners.\nThis includes, but is not limited to, product roadmaps, customer data, internal communications,\nproprietary algorithms, financial information, and unreleased features. Confidential information may not\nbe shared with unauthorized individuals inside or outside the organization. These obligations continue\nafter employment ends.\nWorkplace Conduct Policy\nEmployees must maintain a respectful, profes

In [34]:
all_splits[0].metadata

{'producer': 'ReportLab PDF Library - www.reportlab.com',
 'creator': '(unspecified)',
 'creationdate': '2025-11-20T23:23:16+00:00',
 'author': '(anonymous)',
 'keywords': '',
 'moddate': '2025-11-20T23:23:16+00:00',
 'subject': '(unspecified)',
 'title': '(anonymous)',
 'trapped': '/False',
 'source': 'resources/acmecorp-employee-handbook.pdf',
 'total_pages': 1,
 'page': 0,
 'page_label': '1',
 'start_index': 0}

In [39]:
print(all_splits[0].page_content)

Employee Handbook
Non-Disclosure Agreement (NDA) Policy
Employees must protect confidential information belonging to the company, its clients, and partners.
This includes, but is not limited to, product roadmaps, customer data, internal communications,
proprietary algorithms, financial information, and unreleased features. Confidential information may not
be shared with unauthorized individuals inside or outside the organization. These obligations continue
after employment ends.
Workplace Conduct Policy
Employees must maintain a respectful, professional environment free from harassment, discrimination,
and intimidation. All employees are expected to follow organizational values, collaborate effectively,
and communicate constructively. Disruptive behavior, verbal abuse, or misuse of company systems is
prohibited. Violations may result in disciplinary action.
Paid Time Off (PTO) Policy
Full■time employees accrue PTO according to the following schedule:  0–1 years of service: 10 days


In [ ]:
## chuk size = 1000 meanse num of charecters in the chunk not the words by default anyways you can set the word as well
# def count_words(text):
#     return len(text.split())

# text_splitter = RecursiveCharacterTextSplitter(
#     chunk_size=1000,          # Now means 1000 words
#     chunk_overlap=200,        # Now means 200 words
#     length_function=count_words,
#     add_start_index=True
# )

chk = all_splits[0].page_content
len(chk.split()), len(chk)



(126, 996)

### Insertion in vector db

In [47]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="example_collection",  ## use full for insurting multi docs in same folder
    embedding_function=embeddings,
    persist_directory="C:\E_DRIVE\Langraph-Refresher\database\example_pdf\chroma_db",
)

In [46]:
# Insert documents into the vector store
vector_store.add_documents(documents=all_splits)
print("Insertion complete!")

Insertion complete!


### retrival in vector db

In [48]:
# Fetch all items (or a limited number using the limit parameter)
db_data = vector_store.get(limit=10)

# Extract the IDs list
all_ids = db_data["ids"]
print("First 5 IDs in the DB:", all_ids[:5])

First 5 IDs in the DB: ['fe9a2184-e547-47c3-97cb-ea1df90182bf', 'b1ec550f-bab2-41d4-babc-3a53f4f69c6e', 'bf255a94-70dc-4b9f-ab74-1f09fc32a7ef']


In [ ]:
# Pass a list containing the ID(s) you want to fetch
target_ids = ['794dcc54-b3f3-4563-9ee8-4d1495482de7', 'dc508371-b19b-40c6-93cf-3ca54406cd39']

documents = vector_store.get_by_ids(all_ids)

for doc in documents:
    print(doc)

In [51]:
doc

Document(id='bf255a94-70dc-4b9f-ab74-1f09fc32a7ef', metadata={'source': 'resources/acmecorp-employee-handbook.pdf', 'start_index': 1571, 'creator': '(unspecified)', 'page': 0, 'total_pages': 1, 'subject': '(unspecified)', 'author': '(anonymous)', 'producer': 'ReportLab PDF Library - www.reportlab.com', 'title': '(anonymous)', 'page_label': '1', 'trapped': '/False', 'moddate': '2025-11-20T23:23:16+00:00', 'creationdate': '2025-11-20T23:23:16+00:00', 'keywords': ''}, page_content='business travel. This includes transportation, lodging, meals, and incidental expenses within\nestablished limits. Receipts must be submitted within 14 days of travel. First-class travel, personal\nexpenses, and non-business activities are not reimbursable. Employees should exercise good\njudgment and cost-effective decision-making when traveling on behalf of the company.')

In [52]:
doc.metadata

{'source': 'resources/acmecorp-employee-handbook.pdf',
 'start_index': 1571,
 'creator': '(unspecified)',
 'page': 0,
 'total_pages': 1,
 'subject': '(unspecified)',
 'author': '(anonymous)',
 'producer': 'ReportLab PDF Library - www.reportlab.com',
 'title': '(anonymous)',
 'page_label': '1',
 'trapped': '/False',
 'moddate': '2025-11-20T23:23:16+00:00',
 'creationdate': '2025-11-20T23:23:16+00:00',
 'keywords': ''}

In [53]:
doc.id

'bf255a94-70dc-4b9f-ab74-1f09fc32a7ef'

In [ ]:
results = vector_store.similarity_search("How many days of vacation does an employee get in their first year?")

print(results[0])

In [55]:
results

[Document(id='b1ec550f-bab2-41d4-babc-3a53f4f69c6e', metadata={'source': 'resources/acmecorp-employee-handbook.pdf', 'page': 0, 'author': '(anonymous)', 'total_pages': 1, 'subject': '(unspecified)', 'title': '(anonymous)', 'creationdate': '2025-11-20T23:23:16+00:00', 'keywords': '', 'trapped': '/False', 'moddate': '2025-11-20T23:23:16+00:00', 'page_label': '1', 'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'start_index': 812}, page_content='prohibited. Violations may result in disciplinary action.\nPaid Time Off (PTO) Policy\nFull■time employees accrue PTO according to the following schedule: \x7f 0–1 years of service: 10 days\nper year (0.833 days per month) \x7f 1–3 years of service: 15 days per year (1.25 days per month) \x7f 3+\nyears of service: 20 days per year (1.67 days per month) PTO may be used for vacation, personal\nneeds, or illness. Requests should be submitted in advance through the HR system unless related to\nan emergency. Employe

## croma db as server

In [ ]:
# run server
# chroma run --path "C:\E_DRIVE\Langraph-Refresher\database\example_pdf\chroma_db" --host localhost --port 8000

In [ ]:
# 2. Connect to the running server via HTTP Client
vector_store_server = Chroma(
    collection_name="example_collection", # Must match the exact name you used originally
    embedding_function=embeddings,
    host="localhost",
    port=8000
)


# alternative

# import chromadb
# from langchain_chroma import Chroma

# # Build standard connection
# chroma_client = chromadb.HttpClient(host="localhost", port=8000)

# # Pass it directly to LangChain wrapper
# vector_store = Chroma(
#     client=chroma_client,
#     collection_name="example_collection",
#     embedding_function=embeddings
# )

In [63]:
# Fetch all items (or a limited number using the limit parameter)
db_data = vector_store_server.get(limit=10)

# Extract the IDs list
all_ids = db_data["ids"]
print("First 5 IDs in the DB:", all_ids[:5])

First 5 IDs in the DB: ['fe9a2184-e547-47c3-97cb-ea1df90182bf', 'b1ec550f-bab2-41d4-babc-3a53f4f69c6e', 'bf255a94-70dc-4b9f-ab74-1f09fc32a7ef']


## RAG Agent

In [56]:
from langchain.tools import tool

@tool
def search_handbook(query: str) -> str:
    """Search the employee handbook for information"""
    results = vector_store.similarity_search(query)
    return results[0].page_content

In [57]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[search_handbook],
    system_prompt="You are a helpful agent that can search the employee handbook for information. you have to reply strictly in one line response with answer."
)

In [58]:
response = agent.invoke(
    {"messages": [HumanMessage(content="How many days of vacation does an employee get in their first year?")]}
)

In [59]:
response

{'messages': [HumanMessage(content='How many days of vacation does an employee get in their first year?', additional_kwargs={}, response_metadata={}, id='dbe39c97-3846-4386-a98a-4a59a5fb0065'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'nemotron-3-super:cloud', 'created_at': '2026-06-29T05:38:58.213807804Z', 'done': True, 'done_reason': 'stop', 'total_duration': 10911756798, 'load_duration': None, 'prompt_eval_count': 311, 'prompt_eval_duration': None, 'eval_count': 52, 'eval_duration': None, 'logprobs': None, 'model_name': 'nemotron-3-super:cloud', 'model_provider': 'ollama'}, id='lc_run--019f11eb-9fe8-7981-a9ff-66391398a26a-0', tool_calls=[{'name': 'search_handbook', 'args': {'query': 'vacation days first year'}, 'id': '51a70bd4-e2d9-4837-94ef-99c3ee524bd3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 311, 'output_tokens': 52, 'total_tokens': 363}),
  ToolMessage(content='prohibited. Violations may result in disciplina

In [60]:
print(print(response['messages'][-1].content))

10 days
None
